# 06 — Household-years d'accès : trajectoire rapide contre trajectoire lente

La thèse annonce que la trajectoire rapide délivre **2 648 871 household-years** d'accès sur la
période et la lente **2 566 077**, soit une différence de **82 793**. Aucun fichier du dépôt ne
reproduisait ces chiffres : ce notebook écrit le calcul et vérifie sous quelle convention de
sommation les valeurs annoncées tiennent.

**Définition.** Pour une année donnée, le nombre de ménages ayant accès à l'électricité est
`hh_A + hh_B` (Source A réseau + Source B dispersés desservis). `hh_C` — non desservis — est
exclu par construction. Le household-year est l'intégrale discrète de cette grandeur sur la
période.

**Ancres.** `output/split_abc_projete.csv` ne contient que trois années — 2024, 2035, 2050 —
pour chacune des deux trajectoires (`acces_2035` = rapide, universelle dès 2035 ;
`acces_2050` = lente, tendancielle normalisée pour atteindre 100 % en 2050, cf. notebook 03).
Entre ces ancres, interpolation **linéaire** année par année.

**Deux conventions de sommation** sont évaluées, sans en privilégier une a priori :

| Convention | Bornes | Termes |
|---|---|---|
| A | 2024 → 2050 inclus | 27 |
| B | 2025 → 2050 inclus | 26 |


In [1]:
import os

import numpy as np
import pandas as pd

SPLIT_PATH = os.path.join(os.getcwd(), "output", "split_abc_projete.csv")
HHY_OUT_PATH = os.path.join(os.getcwd(), "output", "household_years_acces.csv")

ANCRES = [2024, 2035, 2050]
TRAJ_LABEL = {"acces_2035": "rapide", "acces_2050": "lente"}

# Valeurs annoncées dans la thèse, à confronter au calcul (aucune n'est imposée au résultat).
THESE = {"rapide": 2_648_871, "lente": 2_566_077, "difference": 82_793}

split = pd.read_csv(SPLIT_PATH)
split["hh_acces"] = split["hh_A"] + split["hh_B"]

# Deux "Santa Rosa" (Beni et Pando) portent le même nom : la clé municipale doit inclure le
# département, sinon np.interp reçoit six ancres au lieu de trois pour ces deux municipios.
split["muni_key"] = split["municipio"] + " (" + split["departamento"] + ")"

assert sorted(split["annee"].unique()) == ANCRES, "les ancres du CSV ne sont pas 2024/2035/2050"
assert set(split["trajectoire"].unique()) == set(TRAJ_LABEL), "trajectoires inattendues"
assert split.groupby(["trajectoire", "annee"])["muni_key"].nunique().nunique() == 1, \
    "nombre de municipios non constant entre trajectoires/années"
assert len(split) == split["muni_key"].nunique() * len(ANCRES) * len(TRAJ_LABEL), \
    "municipios x ancres x trajectoires ne recouvre pas exactement le fichier"

print(f"{split['muni_key'].nunique()} municipios x {len(ANCRES)} ancres x {len(TRAJ_LABEL)} trajectoires "
      f"= {len(split)} lignes")
print(f"noms de municipio distincts : {split['municipio'].nunique()} — "
      f"les homonymes sont désambiguïsés par département")

21 municipios x 3 ancres x 2 trajectoires = 126 lignes
noms de municipio distincts : 20 — les homonymes sont désambiguïsés par département


## 1. Ancres nationales

Somme des 21 municipios. Les deux trajectoires partagent 2024 (même point de départ observé) et
2050 (les deux atteignent 100 % d'accès) ; elles ne diffèrent qu'en 2035, où la rapide a déjà
raccordé tout le monde tandis que la lente laisse un reliquat `hh_C`.

In [2]:
ancres = split.groupby(["trajectoire", "annee"])["hh_acces"].sum().unstack()
ancres_C = split.groupby(["trajectoire", "annee"])["hh_C"].sum().unstack()

print("ménages avec accès (hh_A + hh_B), national :")
print(ancres.round(1).to_string())
print("\nménages non desservis (hh_C), national :")
print(ancres_C.round(1).to_string())

assert np.isclose(ancres.loc["acces_2035", 2024], ancres.loc["acces_2050", 2024]), "2024 doit être commun"
assert np.isclose(ancres.loc["acces_2035", 2050], ancres.loc["acces_2050", 2050]), "2050 doit être commun"
print(f"\n2024 commun aux deux trajectoires : {ancres.loc['acces_2035', 2024]:,.0f} ménages")
print(f"2050 commun aux deux trajectoires : {ancres.loc['acces_2035', 2050]:,.0f} ménages")

ménages avec accès (hh_A + hh_B), national :
annee           2024      2035      2050
trajectoire                             
acces_2035   72820.0  101783.0  120199.0
acces_2050   72820.0   95414.2  120199.0

ménages non desservis (hh_C), national :
annee           2024    2035  2050
trajectoire                       
acces_2035   11389.0     0.0  -0.0
acces_2050   11389.0  6368.7  -0.0

2024 commun aux deux trajectoires : 72,820 ménages
2050 commun aux deux trajectoires : 120,199 ménages


## 2. Interpolation linéaire annuelle

Interpolation menée **par municipio** puis sommée, ce qui est identique à interpoler la somme
nationale (l'interpolation linéaire commute avec la somme dès lors que les ancres sont les mêmes
années pour tous les municipios). L'égalité est vérifiée par assertion plutôt que supposée.

In [3]:
annees = np.arange(2024, 2051)

series_muni = []
for (traj, muni), grp in split.groupby(["trajectoire", "muni_key"]):
    g = grp.sort_values("annee")
    assert list(g["annee"]) == ANCRES, f"ancres incomplètes pour {traj}/{muni}"
    interp = np.interp(annees, g["annee"].values, g["hh_acces"].values)
    series_muni.append(pd.DataFrame({"trajectoire": traj, "muni_key": muni,
                                     "annee": annees, "hh_acces": interp}))
series_muni = pd.concat(series_muni, ignore_index=True)

serie = series_muni.pivot_table(index="annee", columns="trajectoire", values="hh_acces", aggfunc="sum")
serie = serie.rename(columns=TRAJ_LABEL)
serie["ecart"] = serie["rapide"] - serie["lente"]

serie_nat = pd.DataFrame({
    TRAJ_LABEL[t]: np.interp(annees, ANCRES, ancres.loc[t, ANCRES].values) for t in ancres.index
}, index=annees)
assert np.allclose(serie[["rapide", "lente"]].values, serie_nat.values), \
    "interpoler-puis-sommer diffère de sommer-puis-interpoler"

for _t in ancres.index:
    for _a in ANCRES:
        assert np.isclose(serie.loc[_a, TRAJ_LABEL[_t]], ancres.loc[_t, _a]), \
            f"la série interpolée ne repasse pas par l'ancre {_t}/{_a}"

print("série annuelle des ménages avec accès :")
print(serie.round(1).to_string())

série annuelle des ménages avec accès :
trajectoire    rapide     lente   ecart
annee                                  
2024          72820.0   72820.0     0.0
2025          75453.0   74874.0   579.0
2026          78086.0   76928.0  1157.9
2027          80719.0   78982.1  1736.9
2028          83352.0   81036.1  2315.9
2029          85985.0   83090.1  2894.9
2030          88618.0   85144.1  3473.8
2031          91251.0   87198.2  4052.8
2032          93884.0   89252.2  4631.8
2033          96517.0   91306.2  5210.8
2034          99150.0   93360.2  5789.7
2035         101783.0   95414.2  6368.7
2036         103010.7   97066.6  5944.1
2037         104238.4   98718.9  5519.6
2038         105466.2  100371.2  5095.0
2039         106693.9  102023.5  4670.4
2040         107921.6  103675.8  4245.8
2041         109149.4  105328.1  3821.2
2042         110377.1  106980.5  3396.6
2043         111604.8  108632.8  2972.1
2044         112832.6  110285.1  2547.5
2045         114060.3  111937.4  2122.9


## 3. Household-years sous les deux conventions

L'écart annuel `rapide − lente` est nul en 2024 et en 2050 (ancres communes). Il est donc
**identique sous les deux conventions** : le terme 2024 qui sépare A de B vaut zéro dans la
différence. La différence annoncée ne permet à elle seule pas de trancher entre les conventions ;
seuls les niveaux le permettent.

In [4]:
CONVENTIONS = {"A — 2024→2050 inclus (27 termes)": 2024,
               "B — 2025→2050 inclus (26 termes)": 2025}

rows = []
for nom, borne in CONVENTIONS.items():
    bloc = serie.loc[borne:2050]
    assert len(bloc) == 2050 - borne + 1
    rows.append(dict(convention=nom, termes=len(bloc),
                     rapide=bloc["rapide"].sum(), lente=bloc["lente"].sum(),
                     difference=bloc["rapide"].sum() - bloc["lente"].sum()))
hhy = pd.DataFrame(rows).set_index("convention")

print(hhy.round(1).to_string())
print()
for nom, r in hhy.iterrows():
    print(f"{nom} : rapide = {r['rapide']:,.0f}  |  lente = {r['lente']:,.0f}  |  "
          f"différence = {r['difference']:,.0f}")

assert np.isclose(hhy["difference"].iloc[0], hhy["difference"].iloc[1]), \
    "la différence devrait être invariante : l'écart annuel est nul en 2024"
print(f"\nÉcart annuel en 2024 : {serie.loc[2024, 'ecart']:.6f} — la différence est donc "
      f"invariante au choix de convention.")

                                  termes     rapide      lente  difference
convention                                                                
A — 2024→2050 inclus (27 termes)      27  2721690.5  2638897.2     82793.3
B — 2025→2050 inclus (26 termes)      26  2648870.5  2566077.2     82793.3

A — 2024→2050 inclus (27 termes) : rapide = 2,721,691  |  lente = 2,638,897  |  différence = 82,793
B — 2025→2050 inclus (26 termes) : rapide = 2,648,871  |  lente = 2,566,077  |  différence = 82,793

Écart annuel en 2024 : 0.000000 — la différence est donc invariante au choix de convention.


## 4. Confrontation aux valeurs annoncées dans la thèse

In [5]:
print(f"thèse : rapide = {THESE['rapide']:,}  |  lente = {THESE['lente']:,}  |  "
      f"différence = {THESE['difference']:,}\n")

comp = []
for nom, r in hhy.iterrows():
    for cle in ["rapide", "lente", "difference"]:
        comp.append(dict(convention=nom, grandeur=cle, calcule=r[cle], these=THESE[cle],
                         ecart=r[cle] - THESE[cle]))
comp = pd.DataFrame(comp)
comp["reproduit"] = comp["ecart"].abs() < 1.0
print(comp.round(2).to_string(index=False))

conv_ok = [nom for nom, sub in comp.groupby("convention", sort=False) if sub["reproduit"].all()]
print()
if conv_ok:
    print(f"Convention qui reproduit les trois valeurs annoncées (à moins d'un household-year) : "
          f"{conv_ok[0]}")
else:
    print("Aucune convention ne reproduit les valeurs annoncées.")

assert len(conv_ok) <= 1, "les deux conventions ne peuvent pas reproduire les mêmes niveaux"

thèse : rapide = 2,648,871  |  lente = 2,566,077  |  différence = 82,793

                      convention   grandeur    calcule   these    ecart  reproduit
A — 2024→2050 inclus (27 termes)     rapide 2721690.51 2648871 72819.51      False
A — 2024→2050 inclus (27 termes)      lente 2638897.17 2566077 72820.17      False
A — 2024→2050 inclus (27 termes) difference   82793.33   82793     0.33       True
B — 2025→2050 inclus (26 termes)     rapide 2648870.51 2648871    -0.49       True
B — 2025→2050 inclus (26 termes)      lente 2566077.17 2566077     0.17       True
B — 2025→2050 inclus (26 termes) difference   82793.33   82793     0.33       True

Convention qui reproduit les trois valeurs annoncées (à moins d'un household-year) : B — 2025→2050 inclus (26 termes)


## 5. Décomposition de la différence par cluster

Où se logent les household-years gagnés par la trajectoire rapide. La décomposition est menée
sur la convention B, mais elle vaut pour les deux : l'écart annuel 2024 est nul municipio par
municipio, pas seulement au total.

In [6]:
cl = split[["muni_key", "municipio", "cluster"]].drop_duplicates()
assert cl["muni_key"].is_unique, "un muni_key porte deux clusters"

det = series_muni[series_muni["annee"].between(2025, 2050)]
det = det.pivot_table(index="muni_key", columns="trajectoire", values="hh_acces", aggfunc="sum")
det = det.rename(columns=TRAJ_LABEL).reset_index().merge(cl, on="muni_key")
det["difference"] = det["rapide"] - det["lente"]

par_cluster = det.groupby("cluster")[["rapide", "lente", "difference"]].sum()
par_cluster.loc["TOTAL"] = par_cluster.sum()
print("household-years par cluster, convention B (2025→2050) :")
print(par_cluster.round(1).to_string())

print("\nles 8 municipios contribuant le plus à la différence :")
print(det.sort_values("difference", ascending=False)
      [["muni_key", "cluster", "rapide", "lente", "difference"]].head(8).round(1).to_string(index=False))

assert np.isclose(det["difference"].sum(), hhy["difference"].iloc[1]), \
    "la somme des différences par municipio ne recolle pas au total"
print(f"\nrecollement : somme par municipio = {det['difference'].sum():,.1f} household-years")

household-years par cluster, convention B (2025→2050) :
            rapide      lente  difference
cluster                                  
C1        329727.8   311266.8     18461.0
C2         25349.3    23605.4      1743.9
C3       1249579.0  1217367.7     32211.3
C4        536329.5   507734.8     28594.8
C5        507884.8   506102.4      1782.4
TOTAL    2648870.5  2566077.2     82793.3

les 8 municipios contribuant le plus à la différence :


           muni_key cluster   rapide    lente  difference
   Riberalta (Beni)      C3 863379.0 839454.9     23924.1
   Ixiamas (La Paz)      C1 102578.8  94392.5      8186.2
       Sena (Pando)      C4  94323.3  87857.0      6466.3
Guayaramerín (Beni)      C3 319456.6 313449.8      6006.8
       Reyes (Beni)      C1  91658.1  87086.9      4571.2
 Filadelfia (Pando)      C4  84436.0  80395.4      4040.6
  Santa Rosa (Beni)      C1  88278.7  84624.8      3653.8
  San Pedro (Pando)      C4  19253.7  15989.8      3263.9

recollement : somme par municipio = 82,793.3 household-years


## 6. Sauvegarde

Série annuelle interpolée et totaux, pour que les chiffres cités dans la thèse soient traçables
à un fichier du dépôt.

In [7]:
out = serie.reset_index().rename(columns={"rapide": "hh_acces_rapide",
                                          "lente": "hh_acces_lente",
                                          "ecart": "ecart_annuel"})
out["est_ancre"] = out["annee"].isin(ANCRES)
out.to_csv(HHY_OUT_PATH, index=False)

print(f"écrit : {HHY_OUT_PATH} ({len(out)} lignes, {out['est_ancre'].sum()} ancres)")
print()
print("récapitulatif :")
for nom, r in hhy.iterrows():
    print(f"  {nom:<38} rapide {r['rapide']:>12,.0f} | lente {r['lente']:>12,.0f} "
          f"| diff {r['difference']:>8,.0f}")

écrit : C:\Valen\Tfe\bolivia-energy-data\projections\output\household_years_acces.csv (27 lignes, 3 ancres)

récapitulatif :
  A — 2024→2050 inclus (27 termes)       rapide    2,721,691 | lente    2,638,897 | diff   82,793
  B — 2025→2050 inclus (26 termes)       rapide    2,648,871 | lente    2,566,077 | diff   82,793
